# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Missesqueensy/FlyRank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""## My rule

I built a simple baseline rule to prioritize pages that appear to be losing visibility.

The rule gives a higher score to pages that have:
- lower recent impressions than the previous period,
- lower click volume,
- worse average search position.

The goal is to identify pages that may need review before using a machine learning model.

### Reason codes

DECLINING_TRAFFIC
LOW_VISIBILITY
POSITION_DROP"""

'## My rule\n\nI built a simple baseline rule to prioritize pages that appear to be losing visibility.\n\nThe rule gives a higher score to pages that have:\n- lower recent impressions than the previous period,\n- lower click volume,\n- worse average search position.\n\nThe goal is to identify pages that may need review before using a machine learning model.\n\n### Reason codes\n\nDECLINING_TRAFFIC\nLOW_VISIBILITY\nPOSITION_DROP'

In [4]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [5]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [6]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30,
               SUM(f.gsc_clicks)/NULLIF(SUM(f.gsc_impressions),0) AS ctr_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,ctr_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,0.008232
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,0.011735
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,0.000000
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,0.002037
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,0.004334


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import numpy as np

queue = features.copy()

queue["score"] = (
    (queue["imp_prev30"] - queue["imp_last30"]).clip(lower=0)
    + queue["pos_last30"] * 5
    - queue["clk_last30"] * 0.1
)

queue["reason_code"] = np.where(
    queue["imp_last30"] < queue["imp_prev30"],
    "DECLINING_TRAFFIC",
    "LOW_PRIORITY"
)

queue["action"] = np.where(
    queue["score"] > queue["score"].median(),
    "REVIEW",
    "MONITOR"
)

queue = queue.sort_values("score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

queue.head(20)

,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,ctr_last30,score,reason_code,action
30832,client_157ffe4d4a595515,content_9648c4d1595a0794,6001.0,213610.0,12.0,4.520348,0.000291,207630.401738,DECLINING_TRAFFIC,REVIEW
14891,client_1a730cb2640a1abf,content_39e19a3ec2d95f9d,6777.0,178306.0,4.0,35.097900,0.000032,171704.089499,DECLINING_TRAFFIC,REVIEW
37488,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,48447.0,218715.0,146.0,9.494924,0.007737,170300.874620,DECLINING_TRAFFIC,REVIEW
11537,client_23a62021009f63c4,content_65c75874a23fca87,22969.0,191252.0,12.0,30.904741,0.000201,168436.323704,DECLINING_TRAFFIC,REVIEW
53115,client_8ddc46da5414ffd8,content_5913241ddeecf3f1,15058.0,147526.0,88.0,3.283542,0.004976,132475.617711,DECLINING_TRAFFIC,REVIEW
107639,client_23a62021009f63c4,content_e8a52cf3d5988c07,81577.0,196571.0,317.0,11.855581,0.003717,115021.577905,DECLINING_TRAFFIC,REVIEW
67525,client_23a62021009f63c4,content_44f34c0a90047651,27506.0,136464.0,12.0,15.173894,0.000274,109032.669469,DECLINING_TRAFFIC,REVIEW
95883,client_e547b89c05043229,content_21309e9a83c83653,171606.0,272852.0,256.0,4.912553,0.001674,101244.962765,DECLINING_TRAFFIC,REVIEW
68263,client_23a62021009f63c4,content_15168cd1c94e3529,49076.0,145917.0,134.0,5.254497,0.002349,96853.872484,DECLINING_TRAFFIC,REVIEW
69099,client_23a62021009f63c4,content_a9322b74ca7cb1bb,27894.0,109002.0,4.0,18.898033,0.000037,81202.090164,DECLINING_TRAFFIC,REVIEW


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""| Rank | Action  | Why it's here                                        | What would make it wrong?                                                 |
| ---- | ------- | ---------------------------------------------------- | ------------------------------------------------------------------------- |
| 1    | Review  | Large drop in impressions and high average position. | The decline could be caused by seasonality rather than a content problem. |
| 2    | Review  | Lower clicks compared with the previous period.      | Recent indexing delays may temporarily reduce clicks.                     |
| 3    | Review  | Strong decline in visibility score.                  | A recent content update may not have had enough time to take effect.      |
| 4    | Review  | Position worsened while impressions decreased.       | Search demand may have changed instead of page quality.                   |
| 5    | Review  | High baseline score suggests declining performance.  | Missing or incomplete Search Console data could distort the score.        |
| 6    | Monitor | Moderate decline detected.                           | The page may recover naturally without intervention.                      |
| 7    | Monitor | Slight reduction in clicks.                          | Short-term ranking fluctuations may explain the change.                   |
| 8    | Review  | Declining traffic trend.                             | The decline could be due to external events or seasonality.               |
| 9    | Monitor | Lower visibility than previous period.               | Data collection may be incomplete for this period.                        |
| 10   | Review  | Combined decrease in impressions and clicks.         | The page may target a query with declining search demand.                 |
"""

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""## Weak picks

Some pages may receive high scores because of temporary traffic fluctuations rather than long-term decline.

Seasonality, recent indexing changes, or incomplete Google Search Console data may also influence the ranking.

## Leakage check

Only historical features were used.

No future impressions, future clicks, future trend labels, or product flags were included in the baseline score.

The rule is entirely based on information available before the decision date."""

'## Weak picks\n\nSome pages may receive high scores because of temporary traffic fluctuations rather than long-term decline.\n\nSeasonality, recent indexing changes, or incomplete Google Search Console data may also influence the ranking.\n\n## Leakage check\n\nOnly historical features were used.\n\nNo future impressions, future clicks, future trend labels, or product flags were included in the baseline score.\n\nThe rule is entirely based on information available before the decision date.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.